# Bronze Layer — Exploratory Data Analysis

Exploración de los datos en la capa `bronze` para diseñar el job de transformación `bronze → silver`.

Objetivos:
- Detectar inconsistencias de schema entre años (schema drift)
- Definir estrategia de normalización de tipos y nombres de columna
- Evaluar necesidad real de deduplicación
- Validar el join contra la tabla de referencia de zonas (`taxi_zone_lookup`)

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from nyc_taxi_lakehouse.spark.session import create_spark_session
from nyc_taxi_lakehouse.config.settings import settings

spark = create_spark_session("bronze_layer_data_exploration")

## 2. Carga de datos de muestra

Cargamos un mes representativo de cada año (enero 2023, 2024, 2025) para detectar posibles diferencias de schema entre años.

In [ ]:
df_2023 = spark.read.parquet(
    f"s3a://{settings.minio.bucket_name}/{settings.minio.bronze_prefix}/yellow/year=2023/month=01/yellow_tripdata_2023-01.parquet"
)
df_2024 = spark.read.parquet(
    f"s3a://{settings.minio.bucket_name}/{settings.minio.bronze_prefix}/yellow/year=2024/month=01/yellow_tripdata_2024-01.parquet"
)
df_2025 = spark.read.parquet(
    f"s3a://{settings.minio.bucket_name}/{settings.minio.bronze_prefix}/yellow/year=2025/month=01/yellow_tripdata_2025-01.parquet"
)

## 3. Inspección de schema y detección de schema drift

In [ ]:
df_2023.printSchema()

In [ ]:
df_2024.printSchema()

In [ ]:
df_2025.printSchema()

### Hallazgo: schema drift entre años

| Comparación | Cambio |
|---|---|
| 2023 → 2024 | `airport_fee` → `Airport_fee` (cambio de capitalización) |
| 2023 → 2024 | `VendorID`, `passenger_count`, `RatecodeID`, `PULocationID`, `DOLocationID`: cambian de tipo (`long`/`double` → tipos más específicos) |
| 2024 → 2025 | Se agrega la columna `cbd_congestion_fee` (double) — tarifa de congestión CBD vigente desde enero 2025 |

**Total columnas:** 2023: 21 · 2024: 21 · 2025: 22. 20 columnas son comunes a los tres años.

**Conclusión:** el job de transformación necesita normalizar nombres y tipos, y manejar la ausencia de `cbd_congestion_fee` en años anteriores a 2025.

## 4. Normalización de schema

Aplicamos `normalize_schema()` — la función diseñada para resolver el schema drift detectado arriba: renombra columnas a snake_case, castea tipos (enteros para IDs/categorías, `Decimal(10,2)` para montos), agrega `cbd_congestion_fee` cuando falta, y fija un orden canónico de columnas.

In [ ]:
from nyc_taxi_lakehouse.spark.transformations.schema import normalize_schema

df_2023_norm = normalize_schema(df_2023)
df_2024_norm = normalize_schema(df_2024)
df_2025_norm = normalize_schema(df_2025)

In [ ]:
df_2023_norm.printSchema()

In [ ]:
df_2024_norm.printSchema()

In [ ]:
df_2025_norm.printSchema()

**Resultado:** los tres schemas quedan idénticos en nombres, tipos y orden de columnas tras la normalización.

## 5. Análisis de duplicados

Evaluamos si existen filas duplicadas en bronze, y con qué criterio deberían tratarse en silver.

In [ ]:
total_rows = df_2023_norm.count()
distinct_rows = df_2023_norm.dropDuplicates().count()

print(f"Total: {total_rows}")
print(f"Distinct (exact match): {distinct_rows}")
print(f"Exact duplicates: {total_rows - distinct_rows}")

Cero duplicados exactos. Probamos también una clave de negocio parcial (vendor + timestamps + ubicación + tarifa), para descartar duplicados "casi idénticos" con ruido en alguna columna secundaria.

In [ ]:
business_key = [
    "vendor_id",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "pu_location_id",
    "do_location_id",
    "fare_amount",
]

total_rows = df_2023_norm.count()
distinct_by_key = df_2023_norm.dropDuplicates(business_key).count()

print(f"Total: {total_rows}")
print(f"Distinct by business key: {distinct_by_key}")
print(f"Duplicates by business key: {total_rows - distinct_by_key}")

In [ ]:
from pyspark.sql import functions as F

duplicated_keys = (
    df_2023_norm
    .groupBy(*business_key)
    .count()
    .filter(F.col("count") > 1)
)

duplicated_keys.show(5)

In [ ]:
df_2023_norm.join(
    duplicated_keys.select(*business_key),
    on=business_key,
    how="inner",
).orderBy(*business_key).show(10, truncate=False)

### Hallazgo: no son duplicados, son reversiones financieras

Las 165 filas "duplicadas" por clave parcial no son errores de carga: en cada par, los cargos (`mta_tax`, `improvement_surcharge`, `total_amount`, `congestion_surcharge`, `airport_fee`) tienen signo invertido y se cancelan entre sí — es un patrón conocido de NYC TLC que representa una transacción y su reversión/ajuste, no un duplicado real.

**Conclusión:** no hay duplicados reales en los datos. El job de silver aplica `dropDuplicates()` sin argumentos como salvaguarda de idempotencia, sin lógica de negocio adicional (que hubiera eliminado transacciones legítimas).

## 6. Enriquecimiento con lookup table de zonas

Resolvemos `pu_location_id` / `do_location_id` contra `taxi_zone_lookup.csv` para obtener borough, zona y service zone de pickup y dropoff.

In [ ]:
from pyspark.sql.functions import col

lookup_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"s3a://{settings.minio.bucket_name}/{settings.minio.reference_prefix}/taxi_zone_lookup.csv")
)

lookup_df_norm = (
    lookup_df
    .withColumnRenamed("LocationID", "location_id")
    .withColumnRenamed("Borough", "borough")
    .withColumnRenamed("Zone", "zone")
    .withColumn("location_id", col("location_id").cast("int"))
)

lookup_df_norm.printSchema()

In [ ]:
pu_lookup = lookup_df_norm.select(
    col("location_id").alias("pu_location_id"),
    col("borough").alias("pu_borough"),
    col("zone").alias("pu_zone"),
    col("service_zone").alias("pu_service_zone"),
)

do_lookup = lookup_df_norm.select(
    col("location_id").alias("do_location_id"),
    col("borough").alias("do_borough"),
    col("zone").alias("do_zone"),
    col("service_zone").alias("do_service_zone"),
)

df_enriched = (
    df_2023_norm
    .join(pu_lookup, on="pu_location_id", how="left")
    .join(do_lookup, on="do_location_id", how="left")
)

df_enriched.select(
    "pu_location_id", "pu_borough", "pu_zone",
    "do_location_id", "do_borough", "do_zone",
).show()

`how="left"` para no perder filas cuyo `location_id` no tenga match en la lookup table (por ejemplo, códigos "Unknown"/"N/A" reservados oficialmente por NYC TLC).

In [ ]:
unmatched_pu = df_enriched.filter(F.col("pu_borough").isNull()).count()
unmatched_do = df_enriched.filter(F.col("do_borough").isNull()).count()

print(f"Pickup sin match: {unmatched_pu}")
print(f"Dropoff sin match: {unmatched_do}")

**Resultado:** 0 filas sin match en pickup ni dropoff — la lookup table cubre el 100% de los códigos presentes en los datos de viajes.

## 7. Conclusiones

- **Schema drift:** resuelto vía `normalize_schema()` — renombrado a snake_case, cast de tipos, columna faltante rellenada con `null`, orden canónico fijo.
- **Deduplicación:** no se detectaron duplicados reales. `dropDuplicates()` se aplica como salvaguarda de idempotencia únicamente.
- **Enriquecimiento:** join `left` contra `taxi_zone_lookup` en `pu_location_id` y `do_location_id`, con cobertura del 100%.
- Estas decisiones quedaron implementadas en `spark/transformations/schema.py`, `spark/transformations/enrichment.py`, y orquestadas en `spark/jobs/transform_silver.py`.

## 8. Silver verification

Verifico que el job batch de silver escribió correctamente lo esperado.

In [ ]:
df_silver_check = spark.read.parquet(
    f"s3a://{settings.minio.bucket_name}/{settings.minio.silver_prefix}/yellow/year=2023/month=12/yellow_tripdata_2023-12.parquet"
)

df_silver_check.printSchema()
print("Row count:", df_silver_check.count())

### 8.1 Prueba de la validación de calidad

Pruebo `validate_silver_dataframe` contra un mes real antes de integrarla al job.

In [ ]:
from nyc_taxi_lakehouse.spark.quality.silver_validation import validate_silver_dataframe

result = validate_silver_dataframe(df_silver_check)
print(result.success)
print(result.errors)

### 8.2 Investigación: filas con dropoff antes de pickup

La validación detectó filas inválidas. Investigo si es un patrón sistemático o ruido aleatorio.

In [ ]:
from pyspark.sql import functions as f

invalid_rows = df_silver_check.filter(
    f.col("tpep_dropoff_datetime") < f.col("tpep_pickup_datetime")
)

invalid_rows.select(
    "vendor_id", "tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance", "fare_amount"
).show(15, truncate=False)

### 8.3 Prueba del job de gold

Corro `run_transform_gold` sobre un mes específico para validar el fix del filtro de fechas antes de correr el batch completo.

In [ ]:
from nyc_taxi_lakehouse.spark.jobs.transform_gold import run_transform_gold
from nyc_taxi_lakehouse.download.manifest import DatasetFile

dataset = DatasetFile(year=2023, month=1)

run_transform_gold(
    spark=spark,
    dataset=dataset,
    bucket_name=settings.minio.bucket_name,
    silver_prefix=settings.minio.silver_prefix,
    gold_prefix=settings.minio.gold_prefix,
)

### 8.4 Gold verification

Confirmo que el resultado escrito en `gold/` tiene el grano y la estructura esperada.

In [ ]:
df_gold_check = spark.read.parquet(
    f"s3a://{settings.minio.bucket_name}/{settings.minio.gold_prefix}/yellow/year=2023/month=01/yellow_tripdata_2023-01.parquet"
)

df_gold_check.printSchema()
df_gold_check.show(35, truncate=False)
print("Row count:", df_gold_check.count())